# 📈 Valuation Studio: Interactive DCF Walkthrough
This notebook demonstrates how to import the `valuation_studio` quantitative engine into a custom research environment to programmatically evaluate equities without the CLI.

In [1]:
import pandas as pd

from valuation_studio.dcf import DCFEngine
from valuation_studio.loaders import DataLoader
from valuation_studio.scenarios import ScenarioManager
from valuation_studio.statements import FinancialModel

# 1. Instantiate the Engine for NVIDIA
ticker = "NVDA"
print(f"Fetching Live Consensus Data for {ticker}...")
data = DataLoader.from_yfinance(ticker)

print(f"Market Price: ${data.current_price:.2f}")
print(f"Wall St Target: ${data.wall_st_target:.2f}")
print(f"Consensus Year 1 Growth: {data.consensus_growth_y1 * 100:.1f}%")

Fetching Live Consensus Data for NVDA...
Market Price: $210.96
Wall St Target: $301.62
Consensus Year 1 Growth: 85.2%


In [2]:
# 2. Build the Projections (Dynamic Fade)
scenarios = ScenarioManager.get_defaults()
base_case = scenarios["base"]

# The engine will automatically fade the Y1/Y2 consensus down to 2.5% terminal
model = FinancialModel(data)
proj = model.project(
    forecast_years=5,
    rev_growth=[data.consensus_growth_y1, data.consensus_growth_y2],
    ebitda_margin=base_case.ebitda_margin,
)

df_proj = pd.DataFrame(
    {"Revenue ($M)": proj.revenue, "EBITDA ($M)": proj.ebitda, "FCFF ($M)": proj.fcff},
    index=proj.years,
)

display(df_proj.style.format("${:,.1f}"))

,Revenue ($M),EBITDA ($M),FCFF ($M)
2027,"$399,917,176,000.0","$167,965,213,920.0","$97,658,046,875.2"
2028,"$999,193,064,236.0","$419,661,086,979.1","$230,038,238,417.7"
2029,"$2,197,025,709,642.1","$922,750,798,049.7","$517,793,596,397.5"
2030,"$4,304,061,246,217.3","$1,807,705,723,411.3","$1,038,335,019,994.7"
2031,"$7,606,274,764,275.1","$3,194,635,400,995.5","$1,877,119,584,786.8"


In [3]:
# 3. Execute the Unlevered DCF
dcf_res = DCFEngine.value(
    proj=proj,
    wacc=base_case.wacc,
    terminal_growth=base_case.terminal_growth,
    shares_outstanding=data.shares_outstanding,
    current_cash=proj.cash[-1],
    current_debt=proj.debt[-1],
)

upside = (dcf_res.implied_share_price_gordon / data.current_price) - 1.0

print(f"Implied Intrinsic Value: ${dcf_res.implied_share_price_gordon:.2f}")
print(f"Margin of Safety: {upside * 100:+.1f}%")

Implied Intrinsic Value: $988.01
Margin of Safety: +368.3%
